In [1]:
"""
=============================================================================
Phase 4: FMG-WACLU Maximum Intensity & The Master Blend  (SELF-CONTAINED)
=============================================================================
Architecture: FMG-WACLU — Future Met Gating + Wind-Augmented ConvLSTM U-Net
  - WindPhysicsModule  : augments each frame with speed, dir, divergence
  - MetForecaster      : autoregressive future weather prediction (ConvLSTM)
  - ConvLSTM hist enc  : captures spatial-temporal dependencies
  - Gated fusion       : blends historical + future meteorological context
  - UNet + ConvGRU dec : spatial refinement + autoregressive T_out rollout

STEP 1 : Load normalization statistics
STEP 2 : Create datasets  (STRIDE=1)
STEP 3 : Train FMG-WACLU Base Expert from scratch  (Phase 1)
STEP 4 : Pure Quantile Refinement  q=0.90  (Phase 4 PQR)
STEP 5 : Master Blend Inference — 0.25×stable + 0.75×spike + Curvature Correction
=============================================================================
"""

import os, gc, time
os.environ.setdefault('PYTORCH_ALLOC_CONF', 'expandable_segments:True')
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler


# ============================================================================
# CONFIGURATION
# ============================================================================

class Config:
    IS_KAGGLE   = os.path.exists('/kaggle/input')
    DATA_ROOT   = ('/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2'
                   '-pollution-forecasting-iitd/aisehack-theme-2'
                   if IS_KAGGLE else './aisehack-theme-2')
    WORKING_DIR = '/kaggle/working' if IS_KAGGLE else '.'
    RAW_DIR     = os.path.join(DATA_ROOT, 'raw')
    TEST_DIR    = os.path.join(DATA_ROOT, 'test_in')

    MONTHS   = ['APRIL_16', 'JULY_16', 'OCT_16', 'DEC_16']
    FEATURES = ['cpm25', 'q2', 't2', 'u10', 'v10', 'pblh',
                'psfc', 'swdown', 'rain', 'PM25', 'NOx', 'SO2', 'NH3']

    NUM_FEATURES       = len(FEATURES)          # 13
    NUM_INPUT_CHANNELS = NUM_FEATURES + 3       # +wind_speed, hour_sin, hour_cos = 16
    U10_IDX            = FEATURES.index('u10')   # 3
    V10_IDX            = FEATURES.index('v10')   # 4

    MAX_SCALE_FEATURES     = {'PM25', 'NH3', 'SO2', 'NOx'}
    LOG_TRANSFORM_FEATURES = {'rain'}

    INPUT_HOURS  = 10
    OUTPUT_HOURS = 16
    WINDOW_SIZE  = INPUT_HOURS + OUTPUT_HOURS

    H, W = 140, 124

    # FMG-WACLU dims  (reduced for T4 speed: ~4x fewer params)
    ENC_CHANNELS = [32, 64, 128]
    GRU_HIDDEN   = 128
    DEC_CHANNELS = [64, 32, 16]
    DROP_RATE    = 0.1

    EPISODE_STD_THRESHOLD = 2.0
    EPISODE_MIN_CPM       = 1.0

    INFERENCE_BATCH_SIZE = 4
    NUM_TEST_SAMPLES     = 218
    SEED                 = 42


class FMGConfig:
    # ---- Phase 1: Base Expert ----
    STRIDE           = 4        # ~710 samples total → fast run
    BASE_BATCH_SIZE  = 2        # T4 OOM at bs=4; AMP+stride gives speed
    BASE_VAL_SPLIT   = 0.1
    BASE_EPOCHS      = 6
    BASE_LR          = 3e-4
    BASE_WEIGHT_DECAY= 1e-4
    BASE_PATIENCE    = 4
    USE_AMP          = True     # fp16 autocast on T4 → ~1.5-2x speedup

    # ---- Phase 4: PQR ----
    BATCH_SIZE   = 2
    VAL_SPLIT    = 0.1
    EPOCHS       = 8
    LR           = 1e-5
    WEIGHT_DECAY = 1e-4
    PATIENCE     = 5
    QUANTILE     = 0.90   # top-10 % episode focus

    # Inference blend
    STABLE_WEIGHT   = 0.25
    SPIKE_WEIGHT    = 0.75
    CURVE_THRESHOLD = 100.0
    EPS             = 1e-8

    WORKING_DIR = Config.WORKING_DIR
    BASE_CKPT   = os.path.join(Config.WORKING_DIR, 'best_fmg_base.pt')
    PQR_CKPT    = os.path.join(Config.WORKING_DIR, 'best_fmg_pqr.pt')
    NORM_STATS  = os.path.join(Config.WORKING_DIR, 'norm_stats_fmg.npz')

    # If you have a pre-trained base from another notebook, set this path:
    PRETRAINED_BASE = '/kaggle/input/notebooks/kaori02/res2-fmg-base/best_fmg_base.pt'


def seed_everything(seed=Config.SEED):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False


# ============================================================================
# DATA PIPELINE  (identical layout to res1)
# ============================================================================

class NormStats:
    def __init__(self):
        self.mean = {}
        self.std  = {}

    def compute_from_raw(self, raw_dir, months, features):
        for feat in features:
            if feat in Config.MAX_SCALE_FEATURES:
                gmax = 0.0
                for m in months:
                    d = np.load(os.path.join(raw_dir, m, f'{feat}.npy'), mmap_mode='r')
                    gmax = max(gmax, float(np.max(np.abs(d))))
                self.mean[feat] = np.zeros((Config.H, Config.W), np.float32)
                self.std[feat]  = np.full((Config.H, Config.W), max(gmax, 1e-15), np.float32)
            elif feat in Config.LOG_TRANSFORM_FEATURES:
                s, sq, n = 0.0, 0.0, 0
                for m in months:
                    d = np.load(os.path.join(raw_dir, m, f'{feat}.npy'), mmap_mode='r')
                    for t in range(d.shape[0]):
                        x = np.log1p(d[t].astype(np.float64))
                        s += x.sum(); sq += (x**2).sum(); n += x.size
                mu  = s / n
                std = max(np.sqrt(sq / n - mu**2), 1e-6)
                self.mean[feat] = np.full((Config.H, Config.W), mu,  np.float32)
                self.std[feat]  = np.full((Config.H, Config.W), std, np.float32)
            else:
                cnt = 0
                m_acc = np.zeros((Config.H, Config.W), np.float64)
                m2    = np.zeros((Config.H, Config.W), np.float64)
                for m in months:
                    d = np.load(os.path.join(raw_dir, m, f'{feat}.npy'), mmap_mode='r')
                    for t in range(d.shape[0]):
                        x = d[t].astype(np.float64); cnt += 1
                        delta = x - m_acc; m_acc += delta / cnt; m2 += delta * (x - m_acc)
                self.mean[feat] = m_acc.astype(np.float32)
                self.std[feat]  = np.maximum(np.sqrt(m2 / max(cnt-1, 1)).astype(np.float32), 1e-6)
            print(f"  {feat}: done")

    def normalize(self, data, feat):
        if feat in Config.LOG_TRANSFORM_FEATURES:
            data = np.log1p(data)
        return (data - self.mean[feat]) / self.std[feat]

    def save(self, path):
        np.savez(path, features=list(self.mean.keys()),
                 **{f'mean_{k}': v for k, v in self.mean.items()},
                 **{f'std_{k}':  v for k, v in self.std.items()})

    def load(self, path):
        d = np.load(path)
        for feat in list(d['features']):
            self.mean[feat] = d[f'mean_{feat}']
            self.std[feat]  = d[f'std_{feat}']


def _eng_feats(u10, v10, hour_idx):
    ws  = np.sqrt(u10**2 + v10**2).astype(np.float32)
    _, H, W  = u10.shape
    h   = hour_idx.astype(np.float32)
    hs  = np.sin(2*np.pi*h/24.)[:, None, None] * np.ones((1,H,W), np.float32)
    hc  = np.cos(2*np.pi*h/24.)[:, None, None] * np.ones((1,H,W), np.float32)
    return ws, hs, hc


class PM25TrainDataset(Dataset):
    def __init__(self, raw_dir, months, features, norm_stats, stride=1):
        self.features = features; self.ns = norm_stats
        self.samples = []; self.paths = {}
        for mi, m in enumerate(months):
            T = np.load(os.path.join(raw_dir, m, 'cpm25.npy'), mmap_mode='r').shape[0]
            for i in range((T - Config.WINDOW_SIZE) // stride + 1):
                self.samples.append((mi, i*stride))
            for f in features:
                self.paths[(mi, f)] = os.path.join(raw_dir, m, f'{f}.npy')
        print(f"  TrainDataset: {len(self.samples)} samples, stride={stride}")

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        mi, s = self.samples[idx]
        inp, u10n, v10n = [], None, None
        for f in self.features:
            c = np.load(self.paths[(mi,f)], mmap_mode='r')[s:s+Config.INPUT_HOURS].astype(np.float32)
            c = self.ns.normalize(c, f)
            inp.append(c)
            if f == 'u10': u10n = c
            elif f == 'v10': v10n = c
        ws, hs, hc = _eng_feats(u10n, v10n, np.arange(s, s+Config.INPUT_HOURS) % 24)
        inp.extend([ws, hs, hc])
        inputs = np.stack(inp, axis=0)          # (C=16, T=10, H, W)

        cpm = np.load(self.paths[(mi,'cpm25')], mmap_mode='r')
        t_raw  = cpm[s+Config.INPUT_HOURS:s+Config.WINDOW_SIZE].astype(np.float32)
        t_log  = np.log1p(t_raw)
        lf_log = np.log1p(cpm[s+Config.INPUT_HOURS-1].astype(np.float32))
        ctx    = cpm[s:s+Config.INPUT_HOURS].astype(np.float32)
        return (torch.from_numpy(inputs), torch.from_numpy(t_log),
                torch.from_numpy(t_raw), torch.from_numpy(lf_log),
                torch.from_numpy(ctx))


class PM25TestDataset(Dataset):
    def __init__(self, test_dir, features, norm_stats):
        self.features = features; self.ns = norm_stats
        self.data = {f: np.load(os.path.join(test_dir, f'{f}.npy'), mmap_mode='c') for f in features}
        self.n = self.data[features[0]].shape[0]
        print(f"  TestDataset: {self.n} samples")

    def __len__(self): return self.n

    def __getitem__(self, idx):
        inp, u10n, v10n = [], None, None
        for f in self.features:
            c = self.ns.normalize(self.data[f][idx].astype(np.float32), f)
            inp.append(c)
            if f == 'u10': u10n = c
            elif f == 'v10': v10n = c
        ws, hs, hc = _eng_feats(u10n, v10n, np.arange(Config.INPUT_HOURS).astype(np.float32))
        inp.extend([ws, hs, hc])
        inputs  = np.stack(inp, axis=0)
        lf_log  = np.log1p(self.data['cpm25'][idx].astype(np.float32)[-1])
        return torch.from_numpy(inputs), torch.from_numpy(lf_log)


# ============================================================================
# FMG-WACLU ARCHITECTURE
# ============================================================================

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, drop=0.):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False)
        self.gn1   = nn.GroupNorm(min(32, out_ch), out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)
        self.gn2   = nn.GroupNorm(min(32, out_ch), out_ch)
        self.drop  = nn.Dropout2d(drop) if drop > 0 else nn.Identity()
        self.skip  = nn.Conv2d(in_ch, out_ch, 1, bias=False) if in_ch != out_ch else nn.Identity()

    def forward(self, x):
        r = self.skip(x)
        x = F.gelu(self.gn1(self.conv1(x)))
        x = self.drop(x)
        x = F.gelu(self.gn2(self.conv2(x)))
        return x + r


class ConvLSTMCell(nn.Module):
    def __init__(self, in_dim, hid_dim, ks=3):
        super().__init__()
        self.hid = hid_dim
        p = ks // 2
        self.conv = nn.Conv2d(in_dim + hid_dim, 4 * hid_dim, ks, padding=p, bias=True)
        self.gn   = nn.GroupNorm(min(32, 4 * hid_dim), 4 * hid_dim)

    def forward(self, x, state=None):
        B, _, H, W = x.shape
        h = x.new_zeros(B, self.hid, H, W) if state is None else state[0]
        c = x.new_zeros(B, self.hid, H, W) if state is None else state[1]
        gates = self.gn(self.conv(torch.cat([x, h], 1)))
        i, f, g, o = gates.chunk(4, 1)
        c = torch.sigmoid(f)*c + torch.sigmoid(i)*torch.tanh(g)
        h = torch.sigmoid(o)*torch.tanh(c)
        return h, c


class ConvGRUCell(nn.Module):
    def __init__(self, in_dim, hid_dim, ks=3):
        super().__init__()
        self.hid = hid_dim
        p = ks // 2
        self.conv_gates = nn.Conv2d(in_dim+hid_dim, 2*hid_dim, ks, padding=p, bias=True)
        self.conv_cand  = nn.Conv2d(in_dim+hid_dim,   hid_dim, ks, padding=p, bias=True)
        self.gn_g = nn.GroupNorm(min(32, 2*hid_dim), 2*hid_dim)
        self.gn_c = nn.GroupNorm(min(32,   hid_dim),   hid_dim)

    def forward(self, x, h):
        comb  = torch.cat([x, h], 1)
        gates = torch.sigmoid(self.gn_g(self.conv_gates(comb)))
        r, u  = gates.chunk(2, 1)
        cand  = torch.tanh(self.gn_c(self.conv_cand(torch.cat([x, r*h], 1))))
        return (1-u)*h + u*cand


class WindPhysicsModule(nn.Module):
    """Augments (B, C, H, W) with speed, sin-dir, cos-dir, divergence -> (B, C+4, H, W)."""
    def __init__(self, u_idx, v_idx):
        super().__init__()
        self.u_idx = u_idx; self.v_idx = v_idx

    def forward(self, x):
        u   = x[:, self.u_idx:self.u_idx+1]
        v   = x[:, self.v_idx:self.v_idx+1]
        mag = torch.sqrt(u**2 + v**2 + 1e-8)
        sd  = u / (mag + 1e-8)
        cd  = v / (mag + 1e-8)
        du  = F.pad(u[:,:,:,1:] - u[:,:,:,:-1], (0,1))
        dv  = F.pad(v[:,:,1:,:] - v[:,:,:-1,:], (0,0,0,1))
        return torch.cat([x, mag, sd, cd, du+dv], 1)


class MetForecaster(nn.Module):
    """ConvLSTM that autoregressively predicts n_future met frames from history."""
    def __init__(self, in_ch, hid):
        super().__init__()
        self.embed = nn.Sequential(
            nn.Conv2d(in_ch, hid, 3, padding=1, bias=False),
            nn.GroupNorm(min(32, hid), hid), nn.GELU())
        self.cell = ConvLSTMCell(hid, hid)
        self.head = nn.Conv2d(hid, in_ch, 1)

    def forward(self, met_seq, n_future):
        # met_seq: (B, T, C, H, W)
        state = None
        for t in range(met_seq.shape[1]):
            state = self.cell(self.embed(met_seq[:, t]), state)
        preds = []; curr = met_seq[:, -1]
        for _ in range(n_future):
            state = self.cell(self.embed(curr), state)
            curr  = self.head(state[0])
            preds.append(curr)
        return torch.stack(preds, 1)   # (B, n_future, C, H, W)


class FMGWACLUNet(nn.Module):
    """
    FMG-WACLU: Future Met Gating + Wind-Augmented ConvLSTM U-Net.
    Input:  (B, C=16, T=10, H, W)   — same as ResGRUUNet
    Output: (B, T_out=16, H, W)     — log-space delta from last_frame_log
    """
    def __init__(self, cfg=Config):
        super().__init__()
        self.cfg   = cfg
        phys_ch    = cfg.NUM_INPUT_CHANNELS + 4   # 16+4=20
        hid        = cfg.GRU_HIDDEN               # 256
        enc        = cfg.ENC_CHANNELS             # [64,128,256]
        dec        = cfg.DEC_CHANNELS             # [128,64,32]
        dr         = cfg.DROP_RATE

        self.wind    = WindPhysicsModule(cfg.U10_IDX, cfg.V10_IDX)
        self.f_embed = nn.Sequential(
            nn.Conv2d(phys_ch, enc[0], 3, padding=1, bias=False),
            nn.GroupNorm(min(32, enc[0]), enc[0]), nn.GELU())
        self.hist_lstm = ConvLSTMCell(enc[0], hid)

        self.met_fc    = MetForecaster(cfg.NUM_INPUT_CHANNELS, enc[0])
        self.fut_embed = nn.Sequential(
            nn.Conv2d(cfg.NUM_INPUT_CHANNELS, enc[0], 3, padding=1, bias=False),
            nn.GroupNorm(min(32, enc[0]), enc[0]), nn.GELU())
        self.fut_lstm  = ConvLSTMCell(enc[0], hid)

        self.gate_net  = nn.Sequential(
            nn.Conv2d(hid*2, hid, 1, bias=False),
            nn.GroupNorm(min(32, hid), hid), nn.Sigmoid())
        self.fuse_net  = nn.Sequential(
            nn.Conv2d(hid*2, hid, 1, bias=False),
            nn.GroupNorm(min(32, hid), hid), nn.GELU())

        self.pool0 = nn.MaxPool2d(2)
        self.enc1  = ConvBlock(hid,    enc[1], dr)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2  = ConvBlock(enc[1], enc[2], dr)
        self.pool2 = nn.AvgPool2d(2, ceil_mode=True)

        self.bottleneck = ConvBlock(enc[2], hid, dr)
        self.dec_gru    = ConvGRUCell(hid, hid)
        self.gru_proj   = nn.Conv2d(hid, enc[2], 1, bias=False)

        self.up2  = nn.ConvTranspose2d(enc[2], dec[0], 2, stride=2)
        self.dec2 = ConvBlock(dec[0]+enc[2], dec[0], dr)
        self.up1  = nn.ConvTranspose2d(dec[0], dec[1], 2, stride=2)
        self.dec1 = ConvBlock(dec[1]+enc[1], dec[1], dr)
        self.up0  = nn.ConvTranspose2d(dec[1], dec[2], 2, stride=2)
        self.dec0 = ConvBlock(dec[2]+hid,    dec[2], dr)

        self.head = nn.Sequential(
            nn.Conv2d(dec[2], dec[2], 3, padding=1, bias=False),
            nn.GroupNorm(min(32, dec[2]), dec[2]), nn.GELU(),
            nn.Conv2d(dec[2], 1, 1))

        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.GroupNorm):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
        nn.init.zeros_(self.head[-1].weight); nn.init.zeros_(self.head[-1].bias)

    def _pad(self, x, tgt):
        dh = tgt.shape[2]-x.shape[2]; dw = tgt.shape[3]-x.shape[3]
        if dh > 0 or dw > 0: return F.pad(x, [0,dw,0,dh])
        if dh < 0 or dw < 0: return x[:,:,:tgt.shape[2],:tgt.shape[3]]
        return x

    def forward(self, x, last_frame_log):
        B, C, T, H, W = x.shape

        # === 1. Historical ConvLSTM encoding (with wind physics) ===
        hs = None
        for t in range(T):
            f = self.wind(x[:,:,t])    # (B, C+4, H, W)
            f = self.f_embed(f)        # (B, enc[0], H, W)
            hs = self.hist_lstm(f, hs)
        h_hist = hs[0]                 # (B, hid, H, W)

        # === 2. Future meteorological context ===
        met_seq    = x.permute(0, 2, 1, 3, 4)                        # (B, T, C, H, W)
        fut_met    = self.met_fc(met_seq, self.cfg.OUTPUT_HOURS)      # (B, T_out, C, H, W)
        fut_avg    = self.fut_embed(fut_met.mean(1))                   # (B, enc[0], H, W)
        fs         = self.fut_lstm(fut_avg, None)
        h_fut      = fs[0]            # (B, hid, H, W)

        # === 3. Gated fusion ===
        cat    = torch.cat([h_hist, h_fut], 1)
        gate   = self.gate_net(cat)
        h      = gate * h_hist + (1 - gate) * self.fuse_net(cat)

        # === 4. UNet spatial encoding ===
        e0 = h
        e1 = self.enc1(self.pool0(e0))
        e2 = self.enc2(self.pool1(e1))
        bn = self.bottleneck(self.pool2(e2))

        # === 5. ConvGRU temporal decoding ===
        gh = bn; outputs = []
        for _ in range(self.cfg.OUTPUT_HOURS):
            gh  = self.dec_gru(bn, gh)
            d   = self.gru_proj(gh)
            d2  = self.dec2(torch.cat([self._pad(self.up2(d),  e2), e2], 1))
            d1  = self.dec1(torch.cat([self._pad(self.up1(d2), e1), e1], 1))
            d0  = self.dec0(torch.cat([self._pad(self.up0(d1), e0), e0], 1))
            outputs.append(self.head(d0).squeeze(1))
        deltas = torch.stack(outputs, 1)   # (B, T_out, H, W)
        return last_frame_log.unsqueeze(1) + deltas


# ============================================================================
# LOSSES
# ============================================================================

class BaseLoss(nn.Module):
    """Huber + SMAPE for stable Phase 1 base training."""
    EPS = FMGConfig.EPS

    def forward(self, pred_log, target_log, target_raw, lf_log, ctx_raw):
        loss = F.huber_loss(pred_log, target_log, delta=0.5)
        with torch.no_grad():
            pr   = F.relu(torch.expm1(pred_log.clamp(max=15)))
            num  = (pr - target_raw).abs()
            den  = (pr.abs() + target_raw.abs()) * 0.5 + self.EPS
            bm   = ctx_raw.mean(1, keepdim=True)
            bs   = ctx_raw.std(1,  keepdim=True) + self.EPS
            ep   = ((target_raw > bm + Config.EPISODE_STD_THRESHOLD*bs)
                    & (target_raw > Config.EPISODE_MIN_CPM)).float()
            nep  = ep.sum().item()
            ec   = 0.0
            if nep >= 10:
                flat = ep.reshape(-1).bool()
                pe   = pr.reshape(-1)[flat]; ye = target_raw.reshape(-1)[flat]
                pc   = pe - pe.mean(); yc = ye - ye.mean()
                ec   = ((pc*yc).sum() /
                        (((pc**2).sum()+self.EPS).sqrt() *
                         ((yc**2).sum()+self.EPS).sqrt())).item()
        return loss, {'global_smape': (num/den).mean().item(),
                      'episode_smape': ((num/den)*ep).sum().item()/max(nep,1.),
                      'episode_corr': ec, 'n_episodes': nep}


class PurePinballLoss(nn.Module):
    """100 % Pinball q=0.90 — 9× penalty on under-predictions."""
    EPS = FMGConfig.EPS
    Q   = FMGConfig.QUANTILE

    def forward(self, pred_log, target_log, target_raw, lf_log, ctx_raw):
        res  = target_log - pred_log
        loss = torch.mean(torch.max(self.Q * res, (self.Q - 1) * res))
        with torch.no_grad():
            pr   = F.relu(torch.expm1(pred_log.clamp(max=15)))
            num  = (pr - target_raw).abs()
            den  = (pr.abs() + target_raw.abs()) * 0.5 + self.EPS
            bm   = ctx_raw.mean(1, keepdim=True)
            bs   = ctx_raw.std(1,  keepdim=True) + self.EPS
            ep   = ((target_raw > bm + Config.EPISODE_STD_THRESHOLD*bs)
                    & (target_raw > Config.EPISODE_MIN_CPM)).float()
            nep  = ep.sum().item()
            ec   = 0.0
            if nep >= 10:
                flat = ep.reshape(-1).bool()
                pe   = pr.reshape(-1)[flat]; ye = target_raw.reshape(-1)[flat]
                pc   = pe - pe.mean(); yc = ye - ye.mean()
                ec   = ((pc*yc).sum() /
                        (((pc**2).sum()+self.EPS).sqrt() *
                         ((yc**2).sum()+self.EPS).sqrt())).item()
        return loss, {'global_smape': (num/den).mean().item(),
                      'episode_smape': ((num/den)*ep).sum().item()/max(nep,1.),
                      'episode_corr': ec, 'n_episodes': nep}


# ============================================================================
# TRAINING UTILITIES
# ============================================================================

@torch.no_grad()
def validate(model, loader, criterion, device, use_amp=False):
    model.eval(); total, acc, n = 0., {}, 0
    for inp, tlog, traw, lf, ctx in loader:
        inp, tlog, traw, lf, ctx = (
            inp.to(device), tlog.to(device), traw.to(device),
            lf.to(device), ctx.to(device))
        with autocast('cuda', enabled=use_amp):
            plog = model(inp, lf)
            loss, m = criterion(plog.float(), tlog, traw, lf, ctx)
        total += loss.item()
        for k, v in m.items(): acc[k] = acc.get(k, 0.) + v
        n += 1
    for k in acc: acc[k] /= max(n, 1)
    return total / max(n, 1), acc


def train_one_epoch(model, loader, criterion, optimizer, scaler, device, epoch, use_amp):
    model.train(); total, acc, n = 0., {}, 0
    for i, (inp, tlog, traw, lf, ctx) in enumerate(loader):
        inp, tlog, traw, lf, ctx = (
            inp.to(device), tlog.to(device), traw.to(device),
            lf.to(device), ctx.to(device))
        optimizer.zero_grad(set_to_none=True)
        with autocast('cuda', enabled=use_amp):
            plog = model(inp, lf)
            loss, m = criterion(plog.float(), tlog, traw, lf, ctx)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        total += loss.item()
        for k, v in m.items(): acc[k] = acc.get(k, 0.) + v
        n += 1
        if i % 50 == 0:
            print(f"  Ep{epoch} [{i}/{len(loader)}] "
                  f"loss={loss.item():.4f}  "
                  f"ep_smape={m.get('episode_smape',0):.4f}  "
                  f"ep_corr={m.get('episode_corr',0):.4f}")
    for k in acc: acc[k] /= max(n, 1)
    return total / max(n, 1), acc


# ============================================================================
# OOM-SAFE SINGLE-MODEL INFERENCE
# ============================================================================

@torch.no_grad()
def run_inference_single(model, norm_stats, device):
    """Returns raw predictions as (N, H, W, T) float32 array."""
    model.eval(); cfg = Config
    preds   = np.zeros((cfg.NUM_TEST_SAMPLES, cfg.H, cfg.W, cfg.OUTPUT_HOURS), np.float32)
    ds      = PM25TestDataset(cfg.TEST_DIR, cfg.FEATURES, norm_stats)
    loader  = DataLoader(ds, batch_size=cfg.INFERENCE_BATCH_SIZE,
                         shuffle=False, num_workers=0)
    idx = 0
    for bx, blf in loader:
        bx  = bx.to(device); blf = blf.to(device)
        bs  = bx.shape[0]
        with autocast('cuda', enabled=FMGConfig.USE_AMP):
            plog = model(bx, blf).float()
        praw = F.relu(torch.expm1(plog.clamp(max=15)))
        preds[idx:idx+bs] = praw.permute(0,2,3,1).cpu().numpy()
        idx += bs
        if idx % 100 == 0 or idx >= cfg.NUM_TEST_SAMPLES:
            print(f"    {idx}/{cfg.NUM_TEST_SAMPLES}")
        del bx, blf, plog, praw; torch.cuda.empty_cache()
    return preds


# ============================================================================
# MASTER BLEND INFERENCE
# ============================================================================

def run_master_blend_inference(norm_stats, device, base_ckpt, pqr_ckpt):
    """
    OOM-safe dual-model inference:
      1. Load Base (Stable) model -> infer -> store -> delete
      2. Load PQR  (Spike) model  -> infer -> store -> delete
      3. Blend: 0.25 * stable + 0.75 * spike
      4. Curvature Correction on peaks > 100
    """
    cfg = Config; fmg = FMGConfig

    print("\n  [Blend] Pass 1/2 — Stable Base expert ...")
    assert os.path.exists(base_ckpt), f"Base ckpt not found: {base_ckpt}"
    m_stable = FMGWACLUNet(cfg).to(device)
    ck = torch.load(base_ckpt, map_location=device, weights_only=True)
    m_stable.load_state_dict(ck['model_state_dict'])
    print(f"    Loaded base checkpoint: {base_ckpt}")
    p_stable = run_inference_single(m_stable, norm_stats, device)
    del m_stable, ck; gc.collect(); torch.cuda.empty_cache()
    print(f"    Stable range: [{p_stable.min():.1f}, {p_stable.max():.1f}]")

    print("\n  [Blend] Pass 2/2 — PQR Spike expert ...")
    assert os.path.exists(pqr_ckpt), f"PQR ckpt not found: {pqr_ckpt}"
    m_spike = FMGWACLUNet(cfg).to(device)
    ck = torch.load(pqr_ckpt, map_location=device, weights_only=True)
    m_spike.load_state_dict(ck['model_state_dict'])
    print(f"    Loaded PQR checkpoint:  {pqr_ckpt}")
    p_spike = run_inference_single(m_spike, norm_stats, device)
    del m_spike, ck; gc.collect(); torch.cuda.empty_cache()
    print(f"    Spike  range: [{p_spike.min():.1f}, {p_spike.max():.1f}]")

    print(f"\n  [Blend] Combining: {fmg.STABLE_WEIGHT} x stable + {fmg.SPIKE_WEIGHT} x spike ...")
    preds = fmg.STABLE_WEIGHT * p_stable + fmg.SPIKE_WEIGHT * p_spike
    del p_stable, p_spike; gc.collect()
    print(f"    Blended range: [{preds.min():.1f}, {preds.max():.1f}]")

    print("\n  [Blend] Applying Curvature Correction (peaks > 100) ...")
    stretch = 1.0 + (preds / 800.0) ** 2
    preds   = np.where(preds > fmg.CURVE_THRESHOLD, preds * stretch, preds)
    ns      = (preds > fmg.CURVE_THRESHOLD).sum()
    print(f"    Stretched {ns:,} pixels ({100*ns/preds.size:.2f} %)")
    print(f"    Final range: [{preds.min():.1f}, {preds.max():.1f}]  mean={preds.mean():.3f}")
    return preds


# ============================================================================
# TRAINING HELPER
# ============================================================================

def _run_phase(name, model, train_ds, val_ds, criterion, lr, wd,
               batch_size, epochs, patience, ckpt_path, device, use_amp=False):
    """Generic training loop with early stopping on episode_smape."""
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              num_workers=4, pin_memory=True, drop_last=True,
                              persistent_workers=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False,
                              num_workers=4, pin_memory=True,
                              persistent_workers=True)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    scaler    = GradScaler('cuda', enabled=use_amp)

    best_smape = float('inf'); best_ep = 0; pat = 0

    for epoch in range(1, epochs+1):
        t0 = time.time()
        tr_loss, tr_m = train_one_epoch(model, train_loader, criterion, optimizer, scaler, device, epoch, use_amp)
        vl_loss, vl_m = validate(model, val_loader, criterion, device, use_amp)
        ep_smape = vl_m.get('episode_smape', float('inf'))
        ep_corr  = vl_m.get('episode_corr',  0.)

        print(f"\n{name} Epoch {epoch}/{epochs}  ({time.time()-t0:.0f}s)")
        print(f"  Train | loss={tr_loss:.4f}  global={tr_m.get('global_smape',0):.4f}"
              f"  ep_smape={tr_m.get('episode_smape',0):.4f}  ep_corr={tr_m.get('episode_corr',0):.4f}")
        print(f"  Val   | loss={vl_loss:.4f}  global={vl_m.get('global_smape',0):.4f}"
              f"  ep_smape={ep_smape:.4f}  ep_corr={ep_corr:.4f}"
              f"  n_ep={vl_m.get('n_episodes',0):.0f}")

        if ep_smape < best_smape:
            best_smape = ep_smape; best_ep = epoch; pat = 0
            torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                        'val_loss': vl_loss, 'episode_smape': ep_smape,
                        'episode_corr': ep_corr, 'val_metrics': vl_m}, ckpt_path)
            print(f"  -> Saved best  episode_smape={ep_smape:.4f}  ep_corr={ep_corr:.4f}")
        else:
            pat += 1
            print(f"  -> No improvement (best={best_smape:.4f} @ epoch {best_ep})"
                  f"  patience {pat}/{patience}")
            if pat >= patience:
                print(f"\n  Early stopping triggered after {epoch} epochs.")
                break

    return best_smape, best_ep


# ============================================================================
# MAIN
# ============================================================================

def main():
    seed_everything()
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Device: {device}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")

    cfg = Config; fmg = FMGConfig

    # ------------------------------------------------------------------
    # STEP 1: Normalization statistics
    # ------------------------------------------------------------------
    print("\n" + "="*60 + "\nSTEP 1: Loading normalization statistics\n" + "="*60)
    norm_stats = NormStats()
    if os.path.exists(fmg.NORM_STATS):
        print(f"  Loading from {fmg.NORM_STATS}")
        norm_stats.load(fmg.NORM_STATS)
    else:
        # Fallback: try to reuse res1 stats (same features)
        res1_path = '/kaggle/input/notebooks/kaori02/res-1/norm_stats_resgru.npz'
        if os.path.exists(res1_path):
            print(f"  Reusing res1 stats from {res1_path}")
            norm_stats.load(res1_path)
            norm_stats.save(fmg.NORM_STATS)
        else:
            print("  No cache found — recomputing ...")
            norm_stats.compute_from_raw(cfg.RAW_DIR, cfg.MONTHS, cfg.FEATURES)
            norm_stats.save(fmg.NORM_STATS)
    gc.collect()

    # ------------------------------------------------------------------
    # STEP 2: Create datasets  (STRIDE=1)
    # ------------------------------------------------------------------
    print("\n" + "="*60 + f"\nSTEP 2: Creating datasets (STRIDE={fmg.STRIDE})\n" + "="*60)
    full_ds = PM25TrainDataset(cfg.RAW_DIR, cfg.MONTHS, cfg.FEATURES, norm_stats, stride=fmg.STRIDE)
    n_val   = int(len(full_ds) * fmg.BASE_VAL_SPLIT)
    n_train = len(full_ds) - n_val
    train_ds, val_ds = torch.utils.data.random_split(
        full_ds, [n_train, n_val],
        generator=torch.Generator().manual_seed(cfg.SEED))
    print(f"  Train: {n_train}  Val: {n_val}")

    # ------------------------------------------------------------------
    # STEP 3: Train FMG-WACLU Base Expert  (Phase 1)
    # ------------------------------------------------------------------
    print("\n" + "="*60 + "\nSTEP 3: FMG-WACLU Base Expert Training\n" + "="*60)
    model = FMGWACLUNet(cfg).to(device)
    params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  FMG-WACLU parameters: {params:,}")

    # Check for pre-trained base
    if os.path.exists(fmg.PRETRAINED_BASE):
        print(f"  Loading pre-trained base: {fmg.PRETRAINED_BASE}")
        ck = torch.load(fmg.PRETRAINED_BASE, map_location=device, weights_only=True)
        model.load_state_dict(ck['model_state_dict'])
        print(f"  epoch={ck.get('epoch','?')}  ep_smape={ck.get('episode_smape','?')}")
        import shutil
        shutil.copy(fmg.PRETRAINED_BASE, fmg.BASE_CKPT)
        base_smape = ck.get('episode_smape', 0.0)
    elif os.path.exists(fmg.BASE_CKPT):
        print(f"  Resuming base from {fmg.BASE_CKPT}")
        ck = torch.load(fmg.BASE_CKPT, map_location=device, weights_only=True)
        model.load_state_dict(ck['model_state_dict'])
        base_smape = ck.get('episode_smape', 0.0)
    else:
        print(f"  Training from scratch | epochs={fmg.BASE_EPOCHS}  lr={fmg.BASE_LR}  patience={fmg.BASE_PATIENCE}")
        print(f"  Loss = Huber(delta=0.5) — stable base training")
        base_criterion = BaseLoss()
        base_smape, _ = _run_phase(
            "BASE", model, train_ds, val_ds, base_criterion,
            fmg.BASE_LR, fmg.BASE_WEIGHT_DECAY,
            fmg.BASE_BATCH_SIZE, fmg.BASE_EPOCHS, fmg.BASE_PATIENCE,
            fmg.BASE_CKPT, device, use_amp=fmg.USE_AMP)
        print(f"\nBase training complete.  Best Episode SMAPE: {base_smape:.4f}")

    # ------------------------------------------------------------------
    # STEP 4: Pure Quantile Refinement  q=0.90
    # ------------------------------------------------------------------
    print("\n" + "="*60 +
          f"\nSTEP 4: Pure Quantile Refinement  q={fmg.QUANTILE}"
          f"  LR={fmg.LR}  Epochs={fmg.EPOCHS}  Patience={fmg.PATIENCE}\n"
          + "="*60)
    print(f"  Loss = 100 % Pinball(q={fmg.QUANTILE}) — Huber removed")

    # Reload best base checkpoint
    ck = torch.load(fmg.BASE_CKPT, map_location=device, weights_only=True)
    model.load_state_dict(ck['model_state_dict'])
    print(f"  Loaded base checkpoint: {fmg.BASE_CKPT}")
    print(f"  Base ep_smape={ck.get('episode_smape','?')}")

    pqr_criterion = PurePinballLoss()
    pqr_smape, pqr_ep = _run_phase(
        "PQR", model, train_ds, val_ds, pqr_criterion,
        fmg.LR, fmg.WEIGHT_DECAY,
        fmg.BATCH_SIZE, fmg.EPOCHS, fmg.PATIENCE,
        fmg.PQR_CKPT, device, use_amp=fmg.USE_AMP)
    print(f"\nPQR fine-tuning complete.  Best Episode SMAPE: {pqr_smape:.4f} @ epoch {pqr_ep}")

    # ------------------------------------------------------------------
    # STEP 5: Master Blend Inference
    # ------------------------------------------------------------------
    print("\n" + "="*60 + "\nSTEP 5: Master Blend Inference\n" + "="*60)
    print(f"  Stable weight : {fmg.STABLE_WEIGHT}  ({fmg.BASE_CKPT})")
    print(f"  Spike  weight : {fmg.SPIKE_WEIGHT}   ({fmg.PQR_CKPT})")

    del model, train_ds, val_ds, full_ds
    gc.collect(); torch.cuda.empty_cache()

    preds = run_master_blend_inference(norm_stats, device, fmg.BASE_CKPT, fmg.PQR_CKPT)

    out_path = os.path.join(fmg.WORKING_DIR, 'preds.npy')
    np.save(out_path, preds)

    print("\n" + "="*60)
    print("PHASE 4 FMG-WACLU MASTER BLEND COMPLETE")
    print(f"  Base Ckpt   : {fmg.BASE_CKPT}")
    print(f"  PQR Ckpt    : {fmg.PQR_CKPT}")
    print(f"  Predictions : {out_path}  shape={preds.shape}")
    print(f"  Best Ep SMAPE (PQR): {pqr_smape:.4f} @ epoch {pqr_ep}")
    print("="*60)


if __name__ == '__main__':
    main()


Device: cuda
GPU: Tesla T4

STEP 1: Loading normalization statistics
  No cache found — recomputing ...
  cpm25: done
  q2: done
  t2: done
  u10: done
  v10: done
  pblh: done
  psfc: done
  swdown: done
  rain: done
  PM25: done
  NOx: done
  SO2: done
  NH3: done

STEP 2: Creating datasets (STRIDE=4)
  TrainDataset: 710 samples, stride=4
  Train: 639  Val: 71

STEP 3: FMG-WACLU Base Expert Training
  FMG-WACLU parameters: 3,451,057
  Training from scratch | epochs=6  lr=0.0003  patience=4
  Loss = Huber(delta=0.5) — stable base training
  Ep1 [0/319] loss=0.0718  ep_smape=0.2965  ep_corr=0.9404
  Ep1 [50/319] loss=0.0865  ep_smape=0.4490  ep_corr=0.9519
  Ep1 [100/319] loss=0.0526  ep_smape=0.3507  ep_corr=0.9722
  Ep1 [150/319] loss=0.0770  ep_smape=0.4213  ep_corr=0.9391
  Ep1 [200/319] loss=0.0598  ep_smape=0.4257  ep_corr=0.9597
  Ep1 [250/319] loss=0.0638  ep_smape=0.3341  ep_corr=0.8819
  Ep1 [300/319] loss=0.0487  ep_smape=0.2666  ep_corr=0.9691

BASE Epoch 1/6  (156s)
  Trai